En este notebook construiremos un cliente usando la biblioteca `requests` para interactuar con el servidor.

In [53]:
import requests
from requests import Response
import numpy as no
import json
import logging
from typing import Any, Dict, Optional, Union, List

In [14]:
base_url = 'http://localhost:8000'
endpoint = '/predict'

Para consumir el modelo, vamos a agregar el endpont a la URL base para obtener la URL completa.

In [16]:
url_with_endpoint_no_params = base_url + endpoint
url_with_endpoint_no_params

'http://localhost:8000/predict'

### Enviando una solicitud al servidor
#### Creando la función response_from_server
Como recoradtorio, este endpont espera un solicitu HTTP POST. La función `post` es parte de la biblioteca `requests`.

In [60]:
# Configura tu logger al inicio de tu módulo
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")

def response_from_server(
    url: str,
    data: List[List[float]],
    headers: Optional[Dict[str, str]] = None,
    timeout: Union[float, int] = 10,
    verbose: bool = True
) -> Response:
    payload: Dict[str, Any] = {"k": k, "data": data}
    """Hace una solicitud POST al servidor y retorna la respuesta.
    Envía un POST en JSON al servidor y retorna el objeto Response.

    Args:
        url: Endpoint al que se envía la petición.
        payload: Diccionario que se serializa como JSON.
        headers: Cabeceras opcionales (e.g., {'Authorization': 'Bearer ...'}).
        timeout: Tiempo máximo en segundos para bloquear la petición.
        verbose: Habilita logs de información.

    Raises:
        requests.RequestException: Si hay error en la solicitud HTTP.
    """
    
    try:
        response = requests.post(url, json=payload, headers=headers, timeout=timeout)
        response.raise_for_status()
        
    except requests.RequestException as err:
        if verbose:
            logger.error("Error en la solicitud POST a %s: %s", url, err)
        # Propaga la excepción para manejo adicional en quien llame
        raise
    else:
        if verbose:
            if response.status_code == 200:
                logger.info("Solicitud exitosa: %s [200 OK]", url)
            else:
                logger.warning("Status code inesperado %s en %s",
                               response.status_code, url)
        return response

In [35]:
# Payload de prueba: k=3 y dos observaciones en el espacio PCA (2 dimensiones)
payload = {"user": "Cristian", "test": True}


In [57]:
probar = response_from_server("http://127.0.0.1:8000/predict")
probar.text

TypeError: response_from_server() missing 1 required positional argument: 'data'

In [62]:
#from your_module import response_from_server

# 1. Define el payload con la estructura que tu PredictRequest espera
payload = {
    "k": 2,
    "data": [
        [5.1, 3.5, 1.4],
        [4.9, 3.0, 1.4]
    ]
}

# 2. Llamada a la función cliente
try:
    probar = response_from_server(
        url="http://127.0.0.1:8000/predict",
        data=[{"k": 2, "data": [[5.1,3.5,1.4],[4.9,3.0,1.4]]}],
        headers={"Content-Type": "application/json"},  # opcional, requests lo añade de todas formas
        timeout=5,                                      # ajusta según tu necesidad
        verbose=True
    )
except Exception as e:
    print("Fallo en la petición:", e)
else:
    # 3. Inspecciona la respuesta
    print("Status code:", probar.status_code)       # debería ser 200
    print("JSON de respuesta:", probar.json())      # {'k': 2, 'clusters': [...], 'inertia': ...}


Fallo en la petición: name 'k' is not defined


In [51]:
url = "http://localhost:8000/predict"
payload = {
    "k": 3,
    "data": [[1.2, 3.4], [2.5, 6.7]]
}

response = response_from_server(url, payload)

# Leer la respuesta
print("Clusters asignados:", response.json())


2025-08-24 18:47:39,159 [ERROR] __main__: Error en la solicitud POST a http://localhost:8000/predict: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /predict (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x000002746D8A6100>: Failed to establish a new connection: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión'))


ConnectionError: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /predict (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x000002746D8A6100>: Failed to establish a new connection: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión'))

In [38]:
import requests

url = "http://127.0.0.1:8000/predict"
payload = {
    "k": 3,
    "data": [
        [  1.24, -0.47 ],
        [ -0.85,  2.13 ]
    ]
}

resp = requests.post(url, json=payload)
print("Status code:", resp.status_code)
print("Respuesta JSON:", resp.json())


Status code: 500
Respuesta JSON: {'detail': 'Error interno en el servidor'}


In [43]:
# Asumiendo que ya aplicaste nest_asyncio y levantaste Uvicorn
#from tu_modulo import response_from_server

resp = response_from_server(
    "http://127.0.0.1:8000/predict",
    payload,
    verbose=True
)

data = resp.json()
assert data["k"] == 3
assert isinstance(data["clusters"], list) and len(data["clusters"]) == 2
assert isinstance(data["inertia"], float)

print("✔️ La API responde correctamente con tu payload de prueba.")

try:
    response = requests.post(url, json=payload, headers=headers, timeout=timeout)
    # Primero guarda la respuesta cruda
    data = response.json()
    response.raise_for_status()
except requests.HTTPError as err:
    # Muestra el contenido devuelto por FastAPI
    print("HTTPError:", err)
    print("Detalle de validación de FastAPI:", data)
    raise
else:
    return response


2025-08-24 16:29:39,077 [ERROR] __main__: Error en la solicitud POST a http://127.0.0.1:8000/predict: 500 Server Error: Internal Server Error for url: http://127.0.0.1:8000/predict


HTTPError: 500 Server Error: Internal Server Error for url: http://127.0.0.1:8000/predict

In [44]:
import requests

print(requests.get("http://127.0.0.1:8000/debug/pipelines").json())
print(requests.get("http://127.0.0.1:8000/debug/pipelines/3/steps").json())


{'1': ['Kmeans'], '2': ['Kmeans'], '3': ['Kmeans'], '4': ['Kmeans'], '5': ['Kmeans'], '6': ['Kmeans'], '7': ['Kmeans']}
{'detail': 'Not Found'}


In [45]:
import requests

print(requests.get("http://127.0.0.1:8000/debug/pipelines/3/centers_shape").json())


{'centers_shape': [3, 2]}


In [ ]:
import requests

resp = requests.post(
    "http://127.0.0.1:8000/predict",
    json=payload,
    timeout=10
)
print("Status:", resp.status_code)
print("Body  :", resp.text)
